## Лабораторная работа 4

### Салятов Сергей, Подосенов Андрей, M3337

В нашем датасете приведены данные о результатах вступительных экзаменов

#### Задание 1: Распределение суммарного балла

Предположим, что суммарный балл распределен нормально

Гипотезы:

$H_0:$ данные распределены нормально

$H_1:$ данные распределены НЕ нормально

Проверим это вручную с помощью метода согласия Пирсона ($\chi^2$)

In [50]:
import pandas as pd
import numpy as np
from scipy import stats

In [51]:
alpha = 0.05
df = pd.read_csv("exams_dataset.csv")

df['total_score'] = df['math score'] + df['reading score'] + df['writing score']
total_scores = df['total_score'].values
n = len(total_scores)

mu = np.mean(total_scores)
sigma = np.std(total_scores, ddof=1)

print(f"Параметры нормального распределения:")
print(f"Среднее (mu) = {mu:.4f}")
print(f"Стандартное отклонение (sigma) = {sigma:.4f}\n")

num_bins = 10
expected_freq = n / num_bins

quantiles = np.linspace(0, 1, num_bins + 1)
bin_edges = stats.norm.ppf(quantiles, loc=mu, scale=sigma)

observed_freq, _ = np.histogram(total_scores, bins=bin_edges)

chi_sq_stat = 0

for i in range(num_bins):
    O = observed_freq[i]
    E = expected_freq
    component = (O - E) ** 2 / E
    chi_sq_stat += component

df_chi = num_bins - 1 - 2
critical_value = stats.chi2.ppf(1 - alpha, df_chi)

print(f"Степени свободы: {df_chi}")
print(f"Критическое значение для alpha={alpha}: {critical_value:.4f}")
print(f"Статистика Хи-квадрат: {chi_sq_stat:.4f}\n")

if chi_sq_stat > critical_value:
    decision = "Отвергаем H0: распределение НЕ является нормальным"
else:
    decision = "Не отвергаем H0: распределение можно считать нормальным"

print("РЕЗУЛЬТАТ ТЕСТА:")
print(f"Решение: {decision}")
print(f"(Критерий: chi_sq_stat = {chi_sq_stat:.4f} {'>' if chi_sq_stat > critical_value else '<='} critical_value = {critical_value:.4f})")

p_value = 1 - stats.chi2.cdf(chi_sq_stat, df_chi)
print(f"p-value = {p_value:.4f}")
if p_value < alpha:
    print("p-value < alpha → отвергаем H0")
else:
    print("p-value >= alpha → не отвергаем H0")

Параметры нормального распределения:
Среднее (mu) = 202.4040
Стандартное отклонение (sigma) = 45.4738

Степени свободы: 7
Критическое значение для alpha=0.05: 14.0671
Статистика Хи-квадрат: 13.0200

РЕЗУЛЬТАТ ТЕСТА:
Решение: Не отвергаем H0: распределение можно считать нормальным
(Критерий: chi_sq_stat = 13.0200 <= critical_value = 14.0671)
p-value = 0.0716
p-value >= alpha → не отвергаем H0


Теперь проверим с помощью встроенного теста Лиллиефорса (модификация KS-тест Колмогорова, когда математическое ожидание и дисперсия неизвестны)

In [52]:
import statsmodels.stats.diagnostic as smsd
stat, p_value = smsd.lilliefors(total_scores, dist='norm')
print("РЕЗУЛЬТАТЫ ТЕСТА ЛИЛЛИЕФОРСА (модифицированный KS-тест):")
print(f"p-value = {p_value:.6f}")

if p_value < alpha:
    conclusion = "Отвергаем H0: распределение НЕ является нормальным (p < 0.05)"
else:
    conclusion = "Не отвергаем H0: нет оснований считать распределение ненормальным (p >= 0.05)"

print(conclusion)

РЕЗУЛЬТАТЫ ТЕСТА ЛИЛЛИЕФОРСА (модифицированный KS-тест):
p-value = 0.053289
Не отвергаем H0: нет оснований считать распределение ненормальным (p >= 0.05)


#### Задание 2: Одинаковость распределений баллов по математике и чтению

Предположим, что баллы по математике и чтению распределены одианково

Гипотезы:

$H_0:$ баллы по математике и чтению распределены одинаково ($F_X = F_Y$)

$H_1:$ баллы по математике и чтению распределены НЕ одинаково ($F_X \neq F_Y$)

Для начала проведём тесты на независимость:

$H_0$ две выборки независимы

$H_1$ две выборки зависимы

In [53]:
import numpy as np
from scipy import stats
import pandas as pd

math_scores = df['math score'].values
reading_scores = df['reading score'].values

print("="*60)
print("ПРОВЕРКА НЕЗАВИСИМОСТИ ВЫБОРОК")
print("="*60)

pearson_corr, pearson_p = stats.pearsonr(math_scores, reading_scores)
spearman_corr, spearman_p = stats.spearmanr(math_scores, reading_scores)

print(f"\nКорреляция Пирсона: {pearson_corr:.4f}, p-value: {pearson_p:.6f}")
print(f"Корреляция Спирмена: {spearman_corr:.4f}, p-value: {spearman_p:.6f}")

if pearson_p < 0.05:
    print("✓ Есть статистически значимая линейная связь - данные ЗАВИСИМЫ")
else:
    print("× Нет статистически значимой линейной связи - данные могут быть независимы")
    
# 2. Тест хи-квадрат на независимость (для категоризированных данных)
# Разобьем на категории (например, 5 групп по баллам)
math_cat = pd.cut(math_scores, bins=5, labels=False)
reading_cat = pd.cut(reading_scores, bins=5, labels=False)

# Создаем таблицу сопряженности
contingency_table = pd.crosstab(math_cat, reading_cat)

chi2, chi_p, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nХи-квадрат тест на независимость:")
print(f"χ² = {chi2:.4f}, p-value = {chi_p:.6f}, df = {dof}")

if chi_p < 0.05:
    print("✓ Отвергаем H0: данные ЗАВИСИМЫ (есть связь между баллами по математике и чтению)")
else:
    print("× Не отвергаем H0: данные НЕЗАВИСИМЫ (нет связи между баллами)")

ПРОВЕРКА НЕЗАВИСИМОСТИ ВЫБОРОК

Корреляция Пирсона: 0.8384, p-value: 0.000000
Корреляция Спирмена: 0.8340, p-value: 0.000000
✓ Есть статистически значимая линейная связь - данные ЗАВИСИМЫ

Хи-квадрат тест на независимость:
χ² = 960.0741, p-value = 0.000000, df = 16
✓ Отвергаем H0: данные ЗАВИСИМЫ (есть связь между баллами по математике и чтению)


Проверим это вручную с помощью критерия знаков для зависимых выборок. Будем смотреть на медиану разности (если она равна 0, то медианы изначальных выборок равны, а значит, нет оснований предполагать, что распределения выборок отличаются)

In [ ]:
math_scores = df['math score'].values
reading_scores = df['reading score'].values

print("="*60)
print("АНАЛИЗ ДЛЯ ЗАВИСИМЫХ ВЫБОРОК: БАЛЛЫ ПО МАТЕМАТИКЕ И ЧТЕНИЮ")
print("="*60)

differences = math_scores - reading_scores
n = len(differences)

print(f"Размер выборки: {n}")
print(f"Минимум разности: {differences.min():.2f}")
print(f"Максимум разности: {differences.max():.2f}")
print(f"Средняя разность: {differences.mean():.4f}")
print(f"Медианная разность: {np.median(differences):.4f}")

print("\n" + "="*60)
print("1. ТЕСТ ЗНАКОВ (SIGN TEST) - РУЧНОЙ РАСЧЁТ")
print("="*60)

# Убираем нулевые разности
nonzero_diff = differences[differences != 0]
n_nonzero = len(nonzero_diff)

# Считаем знаки
n_plus = np.sum(nonzero_diff > 0)  # математика > чтения
n_minus = np.sum(nonzero_diff < 0)  # математика < чтения

print(f"Ненулевых разностей: {n_nonzero}")
print(f"Математика > чтения: {n_plus} раз")
print(f"Математика < чтения: {n_minus} раз")

# Тестовая статистика
S = min(n_plus, n_minus)
print(f"Тестовая статистика S = min(n⁺, n⁻) = {S}")

# Приближение нормальным распределением (для больших n)
mu_S = n_nonzero / 2
sigma_S = np.sqrt(n_nonzero) / 2
Z_sign = (S - mu_S + 0.5) / sigma_S  # поправка на непрерывность

p_value_sign = 2 * (1 - stats.norm.cdf(abs(Z_sign)))
print(f"Z-статистика (нормальная аппроксимация): {Z_sign:.4f}")
print(f"Критическое значение: {stats.norm.ppf(1 - alpha/2)}")
print(f"p-value (двусторонний): {p_value_sign:.6f}")

if p_value_sign < 0.05:
    print("✓ РЕЗУЛЬТАТ: Отвергаем H₀ - распределения различаются")
else:
    print("× РЕЗУЛЬТАТ: Не отвергаем H₀ - нет оснований считать распределения разными")


АНАЛИЗ ДЛЯ ЗАВИСИМЫХ ВЫБОРОК: БАЛЛЫ ПО МАТЕМАТИКЕ И ЧТЕНИЮ
Размер выборки: 1000
Минимум разности: -29.00
Максимум разности: 21.00
Средняя разность: -3.0360
Медианная разность: -3.0000

1. ТЕСТ ЗНАКОВ (SIGN TEST) - РУЧНОЙ РАСЧЁТ
Ненулевых разностей: 964
Математика > чтения: 361 раз
Математика < чтения: 603 раз
Тестовая статистика S = min(n⁺, n⁻) = 361
Z-статистика (нормальная аппроксимация): -7.7621
Критическое значение: 1.959963984540054
p-value (двусторонний): 0.000000
✓ РЕЗУЛЬТАТ: Отвергаем H₀ - распределения различаются


Теперь проверим тестом Вилкоксона (модифицированный критерий Манна-Уитни) 

In [55]:
print("\n" + "="*60)
print("3. ТЕСТ ЗНАКОВОГО РАНГА ВИЛКОКСОНА - ВСТРОЕННЫЙ")
print("="*60)

# Встроенная функция (сложные расчеты рангов)
wilcoxon_stat, wilcoxon_p = stats.wilcoxon(
    math_scores, 
    reading_scores,
    alternative='two-sided',
    method='approx'  # аппроксимация для больших выборок
)

print(f"Статистика Вилкоксона (T): {wilcoxon_stat:.4f}")
print(f"p-value: {wilcoxon_p:.6f}")

if wilcoxon_p < 0.05:
    print("✓ РЕЗУЛЬТАТ: Отвергаем H₀ - распределения различаются")
    # Определяем направление
    median_math = np.median(math_scores)
    median_reading = np.median(reading_scores)
    if median_math > median_reading:
        print(f"  Медиана математики ({median_math:.2f}) > медианы чтения ({median_reading:.2f})")
    else:
        print(f"  Медиана чтения ({median_reading:.2f}) > медианы математики ({median_math:.2f})")
else:
    print("× РЕЗУЛЬТАТ: Не отвергаем H₀ - нет оснований считать распределения разными")


3. ТЕСТ ЗНАКОВОГО РАНГА ВИЛКОКСОНА - ВСТРОЕННЫЙ
Статистика Вилкоксона (T): 148250.0000
p-value: 0.000000
✓ РЕЗУЛЬТАТ: Отвергаем H₀ - распределения различаются
  Медиана чтения (69.00) > медианы математики (66.00)


#### Задание 3: Верно ли, что прослушавшие подготовительные курсы лучше сдали экзамены?

Предположим, что суммарный балл распределен нормально

Гипотезы:

- $H_0:$ прохождение курсов не повлияло на результат сдачи экзаменов

- $H_1:$ студенты, прошедние подготовительные курсы, сдали экзамены лучше

Используемые тесты: 

- T-test для двух выборок

- Критерий Мана-Уитни


In [56]:
from scipy.stats import t, mannwhitneyu

group1 = df[df['test preparation course'] == 'completed']['total_score'].values
group2 = df[df['test preparation course'] == 'none']['total_score'].values

n1, n2 = len(group1), len(group2)
mean1, mean2 = np.mean(group1), np.mean(group2)
var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)

t_stat = (mean1 - mean2) / np.sqrt(var1 / n1 + var2 / n2)

df_welch = (var1 / n1 + var2 / n2) ** 2 / ((var1 / n1) ** 2 / (n1 - 1) + (var2 / n2) ** 2 / (n2 - 1))

alpha = 0.05
p_value = 1 - t.cdf(t_stat, df=df_welch)
t_crit = t.ppf(1 - alpha, df=df_welch)


print("t-статистика:", t_stat)
print("Степени свободы:", df_welch)
print("Критическое значение t при alpha=0.05:", t_crit)
print("p-value:", p_value)

if p_value < 0.05 or t_stat > t_crit:
    print("H0 отвергается: группа с курсами сдала лучше (статистически значимо).")
else:
    print("Нет оснований отвергать H0: статистически значимой разницы нет.")


import numpy as np
from scipy.stats import mannwhitneyu, norm

n1 = len(group1)
n2 = len(group2)

u, p = mannwhitneyu(group1, group2, alternative='greater')

mu_U = n1 * n2 / 2
sigma_U = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

alpha = 0.05
z_crit = norm.ppf(1 - alpha)

u_crit = mu_U + z_crit * sigma_U

print("U-статистика:", u)
print("p-value:", p)
print("Критическое значение U для alpha=0.05:", u_crit)

if u <= u_crit or p >= 0.05:
    print("U-статистика меньше критического значения → H0 не отвергается")
    print("H1 не подтверждается: статистически значимой разницы нет.")
else:
    print("U-статистика больше критического значения → H0 отвергается")
    print("H1 принимается: прошедшие курсы сдали экзамены лучше.")


t-статистика: 8.559942204819432
Степени свободы: 810.9310162379883
Критическое значение t при alpha=0.05: 1.6467348251504443
p-value: 0.0
H0 отвергается: группа с курсами сдала лучше (статистически значимо).
U-статистика: 152265.0
p-value: 1.0672468653419847e-15
Критическое значение U для alpha=0.05: 124460.42844962666
U-статистика больше критического значения → H0 отвергается
H1 принимается: прошедшие курсы сдали экзамены лучше.
